# Inside the Swarm
Self-contained supervised multi-agent research pipeline. STANDARD runs three independent training seeds. All plots use actual simulator executions; this notebook does not contain precomputed numerical claims. Enable Internet to install dependencies and optionally select a GPU.

In [2]:
import torch, numpy, pandas, matplotlib, scipy, sklearn
print("Dependencies already installed; installation cell skipped for offline validation")

Dependencies already installed; installation cell skipped for offline validation


In [3]:
import base64,zlib,json,os,sys
from pathlib import Path
ROOT=Path('/kaggle/working/swarm-mech-interp') if Path('/kaggle').exists() else Path.cwd()/'swarm-notebook-run'
ROOT.mkdir(parents=True,exist_ok=True)
source=json.loads(zlib.decompress(base64.b64decode('...')))
for name,content in source.items():
    path=ROOT/name; path.parent.mkdir(parents=True,exist_ok=True); path.write_text(content)
for name in ('checkpoints','results'): (ROOT/name).mkdir(exist_ok=True)
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
# To resume a previous Kaggle session, copy its checkpoints directory into ROOT first.


In [4]:
import torch
from environments.task import seed_all
seed_all(0)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODE='quick' # quick: 1 seed / 32 hidden; full: 5 seeds / 64 hidden
print(DEVICE, torch.__version__)

cpu 2.11.0


## Validate the simulation and intervention semantics

In [6]:
import unittest
suite=unittest.defaultTestLoader.discover(str(ROOT/'tests'))
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()

## Train, compare baselines, cache states, and run E1–E8
The orchestrator trains MLP/GNN/GAT; evaluates averaging, global/local oracles and message controls; runs probes, outgoing-message interventions, channel and edge ablations, paired patching, scaling and robustness; and writes all CSVs, figures and checkpoints. It resumes completed model checkpoints or the latest completed training epoch. Evaluation reruns deterministically.

In [8]:
from experiments.run import run
manifest=run(MODE,DEVICE,ROOT)
from experiments.supplement import run as supplement
supplement(ROOT)
exec((ROOT/'paper/build_results.py').read_text(), {'__file__':str(ROOT/'paper/build_results.py')})
print(json.dumps(manifest,indent=2))

Training seed 0 (quick, cpu)
                  mean  std
model                      
average       0.929688  NaN
gat           0.906250  NaN
gnn           0.937500  NaN
local_oracle  0.937500  NaN
mlp           0.929688  NaN
oracle        0.937500  NaN
randomized    0.898438  NaN
shuffled      0.937500  NaN
Supplemental experiments and refreshed figures completed.
Generated manuscript tables and hypothesis status from saved data.
{
  "mode": "quick",
  "config": {
    "seeds": 1,
    "hidden": 32,
    "epochs": 5,
    "steps": 5,
    "batch": 64,
    "validation": 128,
    "test": 128,
    "patience": 4
  },
  "seconds": 10.689651250839233,
  "torch": "2.11.0",
  "python": "3.10.9",
  "device": "cpu",
  "training_distribution": "n=10, connected random ring-backbone graph, reliability=.7 +/- .1, rounds=4",
  "test_seeds": [
    300000
  ],
  "warning": "Training seeds are replicates. QUICK is a smoke run. CIs with one seed are unavailable."
}


In [9]:
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(ROOT/'results/paired_contrasts.csv'))
for path in sorted((ROOT/'results/figures').glob('F*.png')):
    display(Image(filename=str(path),width=800))

         comparison      mean  std  ci_low  ci_high  n_seeds
0         gnn - mlp  0.007812  NaN     NaN      NaN        1
1     gnn - average  0.007812  NaN     NaN      NaN        1
2    gnn - shuffled  0.000000  NaN     NaN      NaN        1
3  gnn - randomized  0.039062  NaN     NaN      NaN        1
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


## Export and final summary
The archive includes raw tables, activation caches, trained checkpoints, source, and manuscript. Confidence intervals use training seeds. QUICK cannot estimate between-seed uncertainty.

In [11]:
import shutil
print((ROOT/'results/summary.txt').read_text())
archive=shutil.make_archive(str(ROOT.parent/'inside-the-swarm-results'),'zip',ROOT)
print('Saved:',archive)
print('Manuscript draft:',ROOT/'paper/main.tex')

                  mean  std
model                      
average       0.929688  NaN
gat           0.906250  NaN
gnn           0.937500  NaN
local_oracle  0.937500  NaN
mlp           0.929688  NaN
oracle        0.937500  NaN
randomized    0.898438  NaN
shuffled      0.937500  NaN
Saved: /private/tmp/swarm-notebook-validation/inside-the-swarm-results.zip
Manuscript draft: /private/tmp/swarm-notebook-validation/swarm-notebook-run/paper/main.tex
